In [ ]:
from pathlib import Path

# Локально репозиторий уже открыт. В Colab/Kaggle загружаем его с GitHub.
hosted = Path("/kaggle/working").is_dir() or Path("/content").is_dir()
if hosted:
    work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
    repo_dir = work_dir / "recsys-rl-sequential"
    if repo_dir.exists():
        !git -C {repo_dir} pull --ff-only
    else:
        !git clone https://github.com/daosistt/recsys-rl-sequential {repo_dir}
    %cd {repo_dir}
    %pip install -q -r requirements.txt


In [1]:
from src import config as project_config

# Kaggle: данные из подключённого dataset, результаты в /kaggle/working/artifacts.
# Colab: данные и результаты на Drive; для чтения данные копируются в /content.
# Локально: папки data и artifacts проекта.
# Свои пути можно задать через RECSYS_DATA_DIR и RECSYS_ARTIFACTS_DIR.
project_config.setup_paths()
from src.config import *
from src.prepare_data import load_yambda_data, train_val_test_split
from src.vocabulary import Vocabulary
from src.dataset import RecDataset
from src.train import plot_loss, train_loop
from src.evaluation import (
    load_model,
    prepare_evaluation_data,
    recommend_top_k,
    evaluate_metrics,
    compare_baselines,
)

print(f"Данные: {DATA_DIR}")
print(f"Результаты: {ARTIFACTS_DIR}")
print(f"Устройство: {device}")


Данные: /home/user/MIPT/recsys-rl-sequential/data
Результаты: /home/user/MIPT/recsys-rl-sequential/artifacts
Устройство: cpu


Reward model: замороженный GPT-2 + линейная регрессия Ridge (L2-регуляризация).

Признаки — скрытое представление последнего токена в последовательности «история + оцениваемый трек». Цель — `played_ratio_pct / 100`, без обрезания значений выше 1. История содержит только предыдущие события.

Используем сохранённое временное разбиение бейзлайна. Берём случайную подвыборку взаимодействий каждой части; целевой трек должен быть в словаре, неизвестные треки в контексте остаются UNK. На train обучаем регрессию, по validation MAE выбираем alpha, test используем только для итоговой оценки. MAE/RMSE выводим в процентных пунктах. Сам GPT-2 не дообучаем.
Сравниваем с общим средним обучающей подвыборки и средним откликом каждого трека по полному train. Для неизвестного трека используем общее среднее полного train. Средние по трекам не используют validation/test. Ridge решается через SVD.


In [2]:
# Запустите ячейку импортов. Заново обучать GPT-2 не нужно.
baseline_dir = ARTIFACTS_DIR / "baseline_t4_1000_minfreq5"
reward_dir = ARTIFACTS_DIR / "reward_linear_minfreq5"
reward_params = REWARD_PARAMS.copy()

reward_gpt, reward_vocab, baseline_config = load_model(baseline_dir, device)
reward_seq_len = reward_gpt.config.n_positions
reward_embedding_dim = reward_gpt.config.n_embd
reward_df = load_yambda_data(size=baseline_config["training"]["data_size"])
reward_train, reward_val, reward_test = train_val_test_split(
    reward_df, **baseline_config["split"],
)

del reward_df
gc.collect()


44

In [3]:
from src.reward_model import (
    clear_memory,
    prepare_reward_features,
    prepare_reward_tabular,
    fit_reward_head,
    fit_reward_catboost,
    evaluate_reward_head,
    save_reward_model,
)


In [4]:
reward_features, reward_targets, reward_items = prepare_reward_features(
    reward_gpt, reward_vocab, reward_train, reward_val, reward_test, reward_params,
)
reward_head = fit_reward_head(reward_features, reward_targets)

# Для следующих моделей нужны только готовые признаки, GPT-2 можно выгрузить.
del reward_gpt
clear_memory()


Reward train:
Выбрано 20000 из 3431203 подходящих событий (всего 3677916)


Reward features:   0%|          | 0/313 [00:00<?, ?it/s]

Reward val:
Выбрано 5000 из 484422 подходящих событий (всего 584811)


Reward features:   0%|          | 0/79 [00:00<?, ?it/s]

Reward test:
Выбрано 5000 из 467582 подходящих событий (всего 619135)


Reward features:   0%|          | 0/79 [00:00<?, ?it/s]

Выбрана alpha=0.1, validation MAE=38.936 п.п.


In [5]:
reward_metrics = evaluate_reward_head(
    reward_head, reward_features, reward_targets, reward_train, reward_items,
)
save_reward_model(
    reward_head, reward_metrics, reward_dir, baseline_dir,
    baseline_config, reward_params, reward_targets["train"],
)
reward_metrics

Модель, конфиг и метрики сохранены в /home/user/MIPT/recsys-rl-sequential/artifacts/reward_linear_minfreq5


MAE (п.п.)  RMSE (п.п.)
Split Model                                 
val   Train mean      42.338769    44.520226
      Track mean      39.140700    43.634096
      GPT-2 + Ridge   38.936168    42.620201
test  Train mean      42.249680    44.425908
      Track mean      39.011006    43.762090
      GPT-2 + Ridge   38.607597    42.581903

In [6]:
# Используем уже рассчитанные признаки: GPT-2 повторно прогонять не нужно.
reward_catboost = fit_reward_catboost(
    reward_features, reward_targets, REWARD_CATBOOST_PARAMS,
)


0:	learn: 0.4243629	test: 0.4222670	best: 0.4222670 (0)	total: 71.1ms	remaining: 35.5s
100:	learn: 0.3790136	test: 0.3919003	best: 0.3919003 (100)	total: 726ms	remaining: 2.87s
200:	learn: 0.3611823	test: 0.3854964	best: 0.3854964 (200)	total: 1.37s	remaining: 2.04s
300:	learn: 0.3461709	test: 0.3823591	best: 0.3823591 (300)	total: 2.01s	remaining: 1.33s
400:	learn: 0.3330600	test: 0.3797917	best: 0.3797728 (399)	total: 2.71s	remaining: 669ms
499:	learn: 0.3220334	test: 0.3784276	best: 0.3783927 (496)	total: 3.36s	remaining: 0us

bestTest = 0.3783927459
bestIteration = 496

Shrink model to first 497 iterations.
CatBoost: сохранено 497 деревьев.


In [7]:
reward_comparison = evaluate_reward_head(
    reward_head, reward_features, reward_targets, reward_train, reward_items,
    extra_heads={"GPT-2 + CatBoost": reward_catboost},
)
save_reward_model(
    reward_catboost, reward_comparison, ARTIFACTS_DIR / "reward_catboost_minfreq5",
    baseline_dir, baseline_config, reward_params, reward_targets["train"],
)
reward_comparison


Модель, конфиг и метрики сохранены в /home/user/MIPT/recsys-rl-sequential/artifacts/reward_catboost_minfreq5


MAE (п.п.)  RMSE (п.п.)
Split Model                                    
val   Train mean         42.338769    44.520226
      Track mean         39.140700    43.634096
      GPT-2 + Ridge      38.936168    42.620201
      GPT-2 + CatBoost   37.839375    41.676972
test  Train mean         42.249680    44.425908
      Track mean         39.011006    43.762090
      GPT-2 + Ridge      38.607597    42.581903
      GPT-2 + CatBoost   37.710618    41.652572

Добавляем к признакам GPT-2 длительность трека, прошлый средний отклик трека и пользователя, число прошлых прослушиваний этого трека пользователем и долю прослушиваний менее 50%.

Все статистики считаются **строго до timestamp текущего события**, события с тем же timestamp исключены. Среднее по треку использует только train. Если прошлых данных по треку нет, берём среднее по предыдущим train-событиям; если прошлого нет вообще — 0.5. Для пользователя без прошлого используем среднее 0.5 и долю коротких прослушиваний 0.0. Длительность трека считается доступным свойством объекта. Текущий played_ratio_pct не используется как признак.

In [8]:
# Те же примеры и признаки GPT-2; добавляем только пять исторических признаков.
reward_tabular = prepare_reward_tabular(
    reward_train, reward_val, reward_test, reward_vocab,
    reward_seq_len, reward_params,
)
reward_enriched_features = {
    split: np.column_stack((reward_features[split], reward_tabular[split]))
    for split in ("train", "val", "test")
}
reward_catboost_enriched = fit_reward_catboost(
    reward_enriched_features, reward_targets, REWARD_CATBOOST_PARAMS,
)


Выбрано 20000 из 3431203 подходящих событий (всего 3677916)
Выбрано 5000 из 484422 подходящих событий (всего 584811)
Выбрано 5000 из 467582 подходящих событий (всего 619135)
0:	learn: 0.4207927	test: 0.4189269	best: 0.4189269 (0)	total: 11.7ms	remaining: 5.82s
100:	learn: 0.3241096	test: 0.3353471	best: 0.3353471 (100)	total: 671ms	remaining: 2.65s
200:	learn: 0.3131530	test: 0.3316630	best: 0.3316539 (197)	total: 1.33s	remaining: 1.98s
300:	learn: 0.3035959	test: 0.3305114	best: 0.3304717 (296)	total: 1.98s	remaining: 1.31s
400:	learn: 0.2947734	test: 0.3299595	best: 0.3298974 (351)	total: 2.62s	remaining: 646ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.3298973909
bestIteration = 351

Shrink model to first 352 iterations.
CatBoost: сохранено 352 деревьев.


In [9]:
reward_enriched_comparison = evaluate_reward_head(
    reward_head, reward_features, reward_targets, reward_train, reward_items,
    extra_heads={"GPT-2 + CatBoost": reward_catboost,
                 "GPT-2 + stats + CatBoost": reward_catboost_enriched},
    extra_features={"GPT-2 + stats + CatBoost": reward_enriched_features},
)
save_reward_model(
    reward_catboost_enriched, reward_enriched_comparison,
    ARTIFACTS_DIR / "reward_catboost_stats_minfreq5",
    baseline_dir, baseline_config, reward_params, reward_targets["train"],
    feature_names=[f"gpt_{i}" for i in range(reward_embedding_dim)] + REWARD_EXTRA_FEATURE_NAMES,
)
reward_enriched_comparison


Модель, конфиг и метрики сохранены в /home/user/MIPT/recsys-rl-sequential/artifacts/reward_catboost_stats_minfreq5


MAE (п.п.)  RMSE (п.п.)
Split Model                                            
val   Train mean                 42.338769    44.520226
      Track mean                 39.140700    43.634096
      GPT-2 + Ridge              38.936168    42.620201
      GPT-2 + CatBoost           37.839375    41.676972
      GPT-2 + stats + CatBoost   32.989839    38.827078
test  Train mean                 42.249680    44.425908
      Track mean                 39.011006    43.762090
      GPT-2 + Ridge              38.607597    42.581903
      GPT-2 + CatBoost           37.710618    41.652572
      GPT-2 + stats + CatBoost   33.089835    39.114508

In [10]:
# Только 5 статистических признаков — без векторов GPT-2.
reward_catboost_tabular = fit_reward_catboost(
    reward_tabular, reward_targets, REWARD_CATBOOST_PARAMS,
)


0:	learn: 0.4207242	test: 0.4188836	best: 0.4188836 (0)	total: 1.37ms	remaining: 683ms
100:	learn: 0.3243421	test: 0.3329268	best: 0.3329268 (100)	total: 114ms	remaining: 449ms
200:	learn: 0.3204354	test: 0.3316942	best: 0.3316259 (193)	total: 228ms	remaining: 338ms
300:	learn: 0.3178278	test: 0.3313518	best: 0.3313333 (296)	total: 337ms	remaining: 223ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.3312464976
bestIteration = 346

Shrink model to first 347 iterations.
CatBoost: сохранено 347 деревьев.


In [11]:
reward_final_comparison = evaluate_reward_head(
    reward_head, reward_features, reward_targets, reward_train, reward_items,
    extra_heads={
        "GPT-2 + CatBoost": reward_catboost,
        "GPT-2 + stats + CatBoost": reward_catboost_enriched,
        "Stats + CatBoost": reward_catboost_tabular,
    },
    extra_features={
        "GPT-2 + stats + CatBoost": reward_enriched_features,
        "Stats + CatBoost": reward_tabular,
    },
)
save_reward_model(
    reward_catboost_tabular, reward_final_comparison,
    ARTIFACTS_DIR / "reward_catboost_tabular_minfreq5",
    baseline_dir, baseline_config, reward_params, reward_targets["train"],
    feature_names=REWARD_EXTRA_FEATURE_NAMES,
)
clear_memory()
reward_final_comparison


Модель, конфиг и метрики сохранены в /home/user/MIPT/recsys-rl-sequential/artifacts/reward_catboost_tabular_minfreq5


MAE (п.п.)  RMSE (п.п.)
Split Model                                            
val   Train mean                 42.338769    44.520226
      Track mean                 39.140700    43.634096
      GPT-2 + Ridge              38.936168    42.620201
      GPT-2 + CatBoost           37.839375    41.676972
      GPT-2 + stats + CatBoost   32.989839    38.827078
      Stats + CatBoost           33.124750    39.032933
test  Train mean                 42.249680    44.425908
      Track mean                 39.011006    43.762090
      GPT-2 + Ridge              38.607597    42.581903
      GPT-2 + CatBoost           37.710618    41.652572
      GPT-2 + stats + CatBoost   33.089835    39.114508
      Stats + CatBoost           33.231199    39.385200